# Button RFI — PPO vs GTO

Compares the trained PPO agent's raise frequency on the Button (when folded to)
against the GTO 6-max 100BB BTN RFI range (~45% of hands).

**How to read the heatmap:**
- Rows = first card rank (A down to 2)
- Cols = second card rank (A down to 2)
- Above diagonal = suited hands, below = offsuit
- Green = PPO raises more than GTO, Red = PPO raises less, White = aligned

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from stable_baselines3 import PPO

from poker_env.env import PokerEnv, MultiAgentRunner
from poker_env.agents.baselines import RandomAgent, HeuristicAgent
from poker_env.agents.base import build_observation, decode_action
from poker_env.game import GameState, Action, ActionType, Street
from poker_env.card import RANKS

## GTO BTN RFI range (6-max, 100BB)

Sourced from published solver outputs. `1` = always raise, `0` = always fold, `0.x` = mixed.

In [ ]:
# 13x13 matrix indexed by [rank1][rank2], rank 12=Ace, 0=2
# Upper triangle (i < j) = suited, lower triangle (i > j) = offsuit
# Diagonal = pocket pairs
# Based on standard GTO 6-max BTN RFI (~45% open)

RANK_NAMES = list('23456789TJQKA')  # index 0=2, 12=A

def make_gto_matrix():
    """Build 13x13 GTO raise frequency matrix for BTN RFI."""
    m = np.zeros((13, 13))

    # Pocket pairs — always raise TT+, mix below
    pair_freq = {
        12: 1.0,  # AA
        11: 1.0,  # KK
        10: 1.0,  # QQ
        9:  1.0,  # JJ
        8:  1.0,  # TT
        7:  1.0,  # 99
        6:  1.0,  # 88
        5:  1.0,  # 77
        4:  1.0,  # 66
        3:  0.9,  # 55
        2:  0.8,  # 44
        1:  0.6,  # 33
        0:  0.5,  # 22
    }
    for r, freq in pair_freq.items():
        m[r][r] = freq

    # Suited hands (upper triangle: row < col means col is higher rank)
    # Stored as m[lower_rank][higher_rank]
    suited = {
        # Ax suited
        (11, 12): 1.0,  # AKs
        (10, 12): 1.0,  # AQs
        (9,  12): 1.0,  # AJs
        (8,  12): 1.0,  # ATs
        (7,  12): 1.0,  # A9s
        (6,  12): 1.0,  # A8s
        (5,  12): 1.0,  # A7s
        (4,  12): 1.0,  # A6s
        (3,  12): 1.0,  # A5s
        (2,  12): 1.0,  # A4s
        (1,  12): 1.0,  # A3s
        (0,  12): 0.8,  # A2s
        # Kx suited
        (10, 11): 1.0,  # KQs
        (9,  11): 1.0,  # KJs
        (8,  11): 1.0,  # KTs
        (7,  11): 1.0,  # K9s
        (6,  11): 0.9,  # K8s
        (5,  11): 0.6,  # K7s
        (4,  11): 0.6,  # K6s
        (3,  11): 0.5,  # K5s
        (2,  11): 0.5,  # K4s
        (1,  11): 0.4,  # K3s
        (0,  11): 0.4,  # K2s
        # Qx suited
        (9,  10): 1.0,  # QJs
        (8,  10): 1.0,  # QTs
        (7,  10): 1.0,  # Q9s
        (6,  10): 0.7,  # Q8s
        (5,  10): 0.5,  # Q7s
        (4,  10): 0.0,  # Q6s
        (3,  10): 0.0,  # Q5s
        (2,  10): 0.4,  # Q4s
        (1,  10): 0.4,  # Q3s
        (0,  10): 0.4,  # Q2s
        # Jx suited
        (8,  9):  1.0,  # JTs
        (7,  9):  1.0,  # J9s
        (6,  9):  0.8,  # J8s
        (5,  9):  0.5,  # J7s
        (4,  9):  0.0,  # J6s
        (3,  9):  0.0,  # J5s
        (2,  9):  0.0,  # J4s
        (1,  9):  0.4,  # J3s
        (0,  9):  0.4,  # J2s
        # Tx suited
        (7,  8):  1.0,  # T9s
        (6,  8):  1.0,  # T8s
        (5,  8):  0.6,  # T7s
        (4,  8):  0.0,  # T6s
        (3,  8):  0.0,  # T5s
        (2,  8):  0.0,  # T4s
        (1,  8):  0.4,  # T3s
        (0,  8):  0.4,  # T2s
        # 9x suited
        (6,  7):  1.0,  # 98s
        (5,  7):  0.9,  # 97s
        (4,  7):  0.5,  # 96s
        (3,  7):  0.0,  # 95s
        (2,  7):  0.0,  # 94s
        # 8x suited
        (5,  6):  1.0,  # 87s
        (4,  6):  0.8,  # 86s
        (3,  6):  0.5,  # 85s
        # 7x suited
        (4,  5):  1.0,  # 76s
        (3,  5):  0.7,  # 75s
        # 6x suited
        (3,  4):  0.8,  # 65s
        (2,  4):  0.5,  # 64s
        # 5x suited
        (2,  3):  0.7,  # 54s
    }
    for (lo, hi), freq in suited.items():
        m[lo][hi] = freq  # upper triangle

    # Offsuit hands (lower triangle: m[higher_rank][lower_rank])
    offsuit = {
        # Ax offsuit
        (12, 11): 1.0,  # AKo
        (12, 10): 1.0,  # AQo
        (12, 9):  1.0,  # AJo
        (12, 8):  1.0,  # ATo
        (12, 7):  0.8,  # A9o
        (12, 6):  0.5,  # A8o
        (12, 5):  0.4,  # A7o
        (12, 4):  0.0,  # A6o
        (12, 3):  0.0,  # A5o
        (12, 2):  0.0,  # A4o
        (12, 1):  0.0,  # A3o
        (12, 0):  0.0,  # A2o
        # Kx offsuit
        (11, 10): 1.0,  # KQo
        (11, 9):  1.0,  # KJo
        (11, 8):  1.0,  # KTo
        (11, 7):  0.7,  # K9o
        (11, 6):  0.0,  # K8o
        (11, 5):  0.0,  # K7o
        (11, 4):  0.0,  # K6o
        (11, 3):  0.0,  # K5o
        (11, 2):  0.0,  # K4o
        (11, 1):  0.0,  # K3o
        (11, 0):  0.0,  # K2o
        # Qx offsuit
        (10, 9):  1.0,  # QJo
        (10, 8):  0.8,  # QTo
        (10, 7):  0.5,  # Q9o
        (10, 6):  0.0,  # Q8o
        # Jx offsuit
        (9,  8):  0.8,  # JTo
        (9,  7):  0.5,  # J9o
        (9,  6):  0.0,  # J8o
        # Tx offsuit
        (8,  7):  0.7,  # T9o
        (8,  6):  0.4,  # T8o
        # 9x offsuit
        (7,  6):  0.5,  # 98o
        (7,  5):  0.0,  # 97o
        # connectors
        (6,  5):  0.4,  # 87o
    }
    for (hi, lo), freq in offsuit.items():
        m[hi][lo] = freq  # lower triangle

    return m

GTO = make_gto_matrix()
print(f"GTO range covers {GTO.mean()*100:.1f}% of combos on average")
print(f"(target ~45% for BTN RFI)")

## Collect PPO decisions at BTN RFI spots

In [ ]:
MODEL_PATH = "../models/best_model"   # EvalCallback saves here
# MODEL_PATH = "../models/ppo_poker_final"  # or use final model

model = PPO.load(MODEL_PATH)
print(f"Loaded model from {MODEL_PATH}")

def is_btn_rfi(state: GameState) -> bool:
    """True if hero is on the button, it's preflop, and everyone folded to them."""
    if state.street != Street.PREFLOP:
        return False
    if state.acting_seat != state.button_seat:
        return False
    # Everyone before hero must have folded (no one else has bet)
    # current_bet == bb_amount means only blinds posted, no raises
    if state.current_bet != state.bb_amount:
        return False
    # All players between BTN and blinds must have folded
    n = len(state.players)
    between = []
    seat = (state.bb_seat + 1) % n
    while seat != state.button_seat:
        between.append(seat)
        seat = (seat + 1) % n
    return all(state.players[s].is_folded for s in between)

def hand_key(hole_cards):
    """Return (rank1, rank2, is_suited) with rank1 >= rank2."""
    r1, r2 = hole_cards[0] // 4, hole_cards[1] // 4
    s1, s2 = hole_cards[0] % 4,  hole_cards[1] % 4
    if r1 < r2:
        r1, r2 = r2, r1
        s1, s2 = s2, s1
    suited = (s1 == s2) and (r1 != r2)
    return r1, r2, suited

def collect_btn_rfi_decisions(n_hands: int = 5000, hero_seat: int = 0):
    """
    Run n_hands and record every BTN RFI decision the PPO agent faces.
    Returns dict mapping (rank1, rank2, suited) -> [raise_or_not, ...]
    """
    opponents = [
        RandomAgent(seat=i, seed=i) for i in range(1, 6)
    ]
    from poker_env.game import PokerGame
    game = PokerGame(n_players=6, starting_stack=1000, sb=5, bb=10, seed=99)
    game.reset_session()

    decisions = defaultdict(list)  # key -> list of bools (True = raised)
    spots_found = 0

    for hand_num in range(n_hands):
        game.start_hand()

        while not game.hand_done:
            state = game.state
            seat = state.acting_seat

            if seat == hero_seat and is_btn_rfi(state):
                # This is the spot we care about
                obs = build_observation(state, hero_seat)
                action_idx, _ = model.predict(obs, deterministic=True)
                action = decode_action(int(action_idx), state)

                raised = action.action_type in (
                    ActionType.RAISE, ActionType.ALL_IN
                )
                key = hand_key(state.players[hero_seat].hole_cards)
                decisions[key].append(raised)
                spots_found += 1
                game.step(action)
            elif seat == hero_seat:
                # Non-BTN-RFI hero decision — just call to keep game moving
                from poker_env.game import Action, ActionType
                call_amt = state.current_bet - state.players[hero_seat].bet_street
                if call_amt == 0:
                    game.step(Action(ActionType.CHECK))
                else:
                    game.step(Action(ActionType.CALL))
            else:
                # Opponents act randomly
                import random
                legal = game.legal_actions()
                game.step(random.choice(legal))

    print(f"Collected {spots_found} BTN RFI decisions across {n_hands} hands")
    return decisions

decisions = collect_btn_rfi_decisions(n_hands=10_000)

## Build PPO raise frequency matrix

In [ ]:
def build_ppo_matrix(decisions):
    """Convert decisions dict to 13x13 raise frequency matrix."""
    m = np.full((13, 13), np.nan)  # nan = no data

    for (r1, r2, suited), outcomes in decisions.items():
        if len(outcomes) < 3:  # skip combos with too few samples
            continue
        freq = np.mean(outcomes)

        if r1 == r2:  # pocket pair — diagonal
            m[r1][r2] = freq
        elif suited:  # suited — upper triangle (lower rank row, higher rank col)
            m[r2][r1] = freq
        else:  # offsuit — lower triangle (higher rank row, lower rank col)
            m[r1][r2] = freq

    return m

PPO_MATRIX = build_ppo_matrix(decisions)

# Count combos with data
has_data = ~np.isnan(PPO_MATRIX)
print(f"Hand combos with data: {has_data.sum()} / 91")
valid = PPO_MATRIX[has_data]
print(f"PPO overall raise frequency: {valid.mean()*100:.1f}%")
print(f"GTO overall raise frequency: {GTO[has_data].mean()*100:.1f}%")

## Heatmap: PPO raise frequency

In [ ]:
RANK_LABELS = list(reversed(RANK_NAMES))  # A down to 2 for display

def plot_matrix(matrix, title, cmap='RdYlGn', vmin=0, vmax=1, annot=True):
    # Reorder to show A top-left (rank 12 = index 0 in display)
    display = matrix[::-1, ::-1].T  # flip both axes for standard hand chart orientation

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        display,
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        annot=annot,
        fmt='.0%' if annot else '',
        linewidths=0.5,
        linecolor='#cccccc',
        cbar_kws={'label': 'Raise frequency'},
        xticklabels=RANK_LABELS,
        yticklabels=RANK_LABELS,
        square=True,
    )
    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlabel('Second card rank')
    ax.set_ylabel('First card rank')

    # Label regions
    ax.text(0.02, 0.98, 'Suited ▲', transform=ax.transAxes,
            fontsize=9, va='top', color='#333333')
    ax.text(0.98, 0.02, 'Offsuit ▼', transform=ax.transAxes,
            fontsize=9, ha='right', color='#333333')
    plt.tight_layout()
    return fig

fig1 = plot_matrix(PPO_MATRIX, 'PPO Agent — BTN RFI Raise Frequency')
plt.savefig('ppo_btn_rfi.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmap: GTO range

In [ ]:
fig2 = plot_matrix(GTO, 'GTO — BTN RFI Range (6-max 100BB)')
plt.savefig('gto_btn_rfi.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmap: Deviation (PPO − GTO)

Green = PPO over-raises vs GTO. Red = PPO under-raises (folds too much).

In [ ]:
DEVIATION = PPO_MATRIX - GTO

fig3 = plot_matrix(
    DEVIATION,
    'PPO − GTO Deviation (BTN RFI)',
    cmap='RdYlGn',
    vmin=-1,
    vmax=1,
)
plt.savefig('deviation_btn_rfi.png', dpi=150, bbox_inches='tight')
plt.show()

## Top deviations — biggest leaks

In [ ]:
def combo_name(r1, r2, suited):
    """e.g. (12, 11, True) -> 'AKs'"""
    if r1 == r2:
        return RANK_NAMES[r1] * 2
    high, low = max(r1, r2), min(r1, r2)
    suffix = 's' if suited else 'o'
    return f"{RANK_NAMES[high]}{RANK_NAMES[low]}{suffix}"

rows = []
for (r1, r2, suited), outcomes in decisions.items():
    if len(outcomes) < 3:
        continue
    ppo_freq = np.mean(outcomes)

    if r1 == r2:
        gto_freq = GTO[r1][r2]
    elif suited:
        gto_freq = GTO[r2][r1]
    else:
        gto_freq = GTO[r1][r2]

    deviation = ppo_freq - gto_freq
    rows.append({
        'Hand': combo_name(r1, r2, suited),
        'PPO': f"{ppo_freq:.0%}",
        'GTO': f"{gto_freq:.0%}",
        'Deviation': f"{deviation:+.0%}",
        '_dev': deviation,
        'Samples': len(outcomes),
    })

rows.sort(key=lambda x: abs(x['_dev']), reverse=True)

print(f"{'Hand':<8} {'PPO':>6} {'GTO':>6} {'Deviation':>10} {'Samples':>8}")
print('-' * 45)
for r in rows[:20]:
    print(f"{r['Hand']:<8} {r['PPO']:>6} {r['GTO']:>6} {r['Deviation']:>10} {r['Samples']:>8}")

## Summary

In [ ]:
over_raises  = [(r, d) for r in rows for d in [r['_dev']] if d >  0.15]
under_raises = [(r, d) for r in rows for d in [r['_dev']] if d < -0.15]

print(f"Hands PPO significantly over-raises vs GTO (>15%):")
for r, _ in sorted(over_raises, key=lambda x: -x[1]):
    print(f"  {r['Hand']}: PPO={r['PPO']} GTO={r['GTO']}")

print(f"\nHands PPO significantly under-raises vs GTO (>15% below):")
for r, _ in sorted(under_raises, key=lambda x: x[1]):
    print(f"  {r['Hand']}: PPO={r['PPO']} GTO={r['GTO']}")